# Backtest — MACross + protective stop-loss sensitivity

Companion to `ma_cross.ipynb` focused on the **protective stop loss** knob.  Same MA-crossover strategy across all 6 MA types — the difference is that this notebook defaults to `STOP_PCT = 0.05` (5% protective stop, isolated-margin equivalent at 20× leverage) and adds two stop-loss-specific sections under §4:

- **§4.5 Stop-loss sensitivity sweep** — fix the best Stage 1 MA combo, sweep `STOP_PCTS` (a range of stop-loss percentages plus `None` for the no-stop baseline), and plot PnL + max-drawdown vs the chosen stop_pct.  Reveals the trade-off: tight stops cap losses but cause more whipsaws; loose stops let drawdowns deepen but capture more winners.
- **§4.6 Close-cause table** — for the chosen STOP_PCT (single-config from §2), counts how many fills closed via natural strategy exit vs protective stop vs cross-margin liquidation stop.  Validates that the protective stop is doing the work it should and that liq is acting as a backstop only.

The full v2 analysis stack (drawdown anatomy, regime breakdown, baselines, bootstrap CI, v2 tearsheet, sortable HTML sweep, alt-instrument check, fee sensitivity) is preserved from the parent template.

All metrics derived from realized PnL on closed positions and event-time balance snapshots — no Sharpe / Sortino / Vol / Returns Profit Factor anywhere on the page.  See [`docs/ANALYZER_RETURNS_CAVEAT.md`](../../docs/ANALYZER_RETURNS_CAVEAT.md) for the methodology audit and [`docs/LIQUIDATION_AND_SIZING.md`](../../docs/LIQUIDATION_AND_SIZING.md) §"Protective stop loss" for the design.

## 1. Setup

Imports, shared config (instrument / interval / params / sizing / liquidation), and one-shot data load.

### 1.1 Imports & shared config

All tuneable values live here.  Change once, used everywhere below.

In [ ]:
# ── Imports ────────────────────────────────────────────────────
from decimal import Decimal

from IPython.display import HTML, Markdown, display
from nautilus_trader.model.data import BarType
from nautilus_trader.model.identifiers import Venue

from src.backtesting import make_engine, run_sweep
from src.backtesting.analysis import (
    performance_by_regime, performance_by_year, run_fee_sweep, tag_regimes,
)
from src.backtesting.engine import resolve_strategy_liquidation_config
from src.backtesting.metrics import bootstrap_total_pnl
from src.core import (
    PROTECTIVE_STOP_TAG,
    TOPIC_ACCOUNT_LIQUIDATED,
    TOPIC_POSITION_LIQUIDATED,
    LiquidationConfig,
    SizingConfig,
    bar_type_str,
    get_venue_config,
)
from src.config.settings import get_settings
from src.strategies.ma_cross import MACross, MACrossConfig

# Add notebooks/ to sys.path so charts.py + utils.py imports resolve.
import sys
sys.path.insert(0, str(__import__("pathlib").Path("..").resolve()))

from charts import (
    # backtest charts
    plot_ma_cross,
    plot_equity_curve,
    plot_drawdown_distribution,
    plot_trade_distributions,
    plot_yearly_breakdown,
    plot_baselines_comparison,
    plot_bootstrap_pnl,
    # sweep charts
    plot_pnl_heatmap,
    plot_fee_sensitivity,
    # text + HTML reports
    print_summary_stats,
    print_yearly_breakdown,
    generate_backtest_html,
    generate_sweep_html,
    generate_v2_tearsheet,
)
from utils import (
    # data + setup
    load_backtest_data,
    make_instrument_id,
    print_setup_summary,
    print_liquidation_resolution,
    # diagnostics + analysis
    baselines_for_strategy,
    print_baselines_verdict,
    print_run_diagnostics,
    print_sweep_liquidation_diagnostics,
    # snapshotting
    save_notebook_snapshot,
    # close-cause classification (used by chart + tearsheet)
    classify_position_exits,
    find_account_liq_culprit,
)


# ─────────────────────────────────────────────────────────────────
# Shared config — edit these, the rest of the notebook follows.
# ─────────────────────────────────────────────────────────────────

# All defaults flow from src/config/settings.py (driven by .env), so a value
# validated here in backtest deploys with the *same* number to sandbox + live.
# Override any line below for a one-off experiment — settings stay intact.
settings         = get_settings()

# Data + venue
DATA_SOURCE      = settings.data_source       # default: BINANCE_PERP
EXEC_VENUE       = settings.exec_venue        # default: HYPERLIQUID_PERP
ASSET            = "BTC"                       # backtest-specific (vary per run)
BAR_INTERVAL     = settings.bar_interval       # default: 4h
DATE_START       = None                 # e.g. "2025-01-01" — None means "from start of data"
DATE_END         = None                 # e.g. "2025-04-01" — None means "to end of data"

# Account + leverage — single source of truth in src/config/settings.py.
STARTING_CAPITAL = settings.starting_capital   # default: 1000
TRADE_SIZE       = int(settings.trade_notional)  # default: 2000
LEVERAGE         = settings.leverage            # default: 20

# Strategy + sweep grid
MA_TYPE          = "EMA"                # one of: EMA, SMA, HMA, DEMA, AMA, VIDYA
FAST_MA          = 10                   # primary-config fast MA period
SLOW_MA          = 40                   # primary-config slow MA period
# Per-MA-type sweep grids.  EMA/SMA/DEMA can run long lookbacks
# cleanly so they get a 12×12 grid up to 100 fast / 200 slow.
# HMA/AMA/VIDYA are heavier or less stable at long windows so
# they get a tighter 11×10 grid capped at 50 fast / 100 slow.
_FAST_GRIDS = {
    "EMA":   [5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 75, 100],
    "SMA":   [5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 75, 100],
    "DEMA":  [5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 75, 100],
    "HMA":   [5, 8, 10, 12, 15, 20, 25, 30, 35, 40, 50],
    "AMA":   [5, 8, 10, 12, 15, 20, 25, 30, 35, 40, 50],
    "VIDYA": [5, 8, 10, 12, 15, 20, 25, 30, 35, 40, 50],
}
_SLOW_GRIDS = {
    "EMA":   [10, 15, 20, 25, 30, 35, 40, 45, 50, 75, 100, 200],
    "SMA":   [10, 15, 20, 25, 30, 35, 40, 45, 50, 75, 100, 200],
    "DEMA":  [10, 15, 20, 25, 30, 35, 40, 45, 50, 75, 100, 200],
    "HMA":   [10, 15, 20, 25, 30, 35, 40, 50, 75, 100],
    "AMA":   [10, 15, 20, 25, 30, 35, 40, 50, 75, 100],
    "VIDYA": [10, 15, 20, 25, 30, 35, 40, 50, 75, 100],
}
FAST_PERIODS     = sorted(set(_FAST_GRIDS[MA_TYPE] + [FAST_MA]))
SLOW_PERIODS     = sorted(set(_SLOW_GRIDS[MA_TYPE] + [SLOW_MA]))

# Off-grid "spotlight" combos mixed into the sweep alongside the regular
# fast/slow grid above.  Each entry is a param dict with `_kind: "spotlight"`
# (and optional `_note: ...`).  The `_kind` tag passes through to the sweep
# parquet but is stripped before the strategy_factory sees the params; the
# heatmap (section 4.2) silently excludes spotlight rows so the grid stays
# clean, while the sortable HTML report (section 4.4) badges them with [SPOT].
#
# Per-MA-type spotlights:
#   • EMA    — the trader-lore pairs (8/21, 12/26 MACD, Fibonacci, 9/18)
#   • SMA    — the classic 50/200 golden cross
#   • HMA/DEMA/AMA/VIDYA — empty by default (no widely-used named combos;
#     AMA/VIDYA are adaptive so fixed pairs are meaningless).  Edit the
#     dict below to add ad-hoc combos for any MA type.
_SPOTLIGHTS: dict[str, list[dict]] = {
    "EMA": [
        {"fast": 9,  "slow": 18, "_kind": "spotlight", "_note": "9/18 trial"},
        {"fast": 8,  "slow": 21, "_kind": "spotlight", "_note": "8/21 EMA classic"},
        {"fast": 13, "slow": 21, "_kind": "spotlight", "_note": "Fibonacci"},
        {"fast": 12, "slow": 26, "_kind": "spotlight", "_note": "MACD periods"},
    ],
    "SMA": [
        {"fast": 50, "slow": 200, "_kind": "spotlight", "_note": "Golden cross"},
    ],
    "HMA":   [],
    "DEMA":  [],
    "AMA":   [],
    "VIDYA": [],
}
SPOTLIGHT_COMBOS: list[dict] = _SPOTLIGHTS[MA_TYPE]

# Sizing — None falls back to fixed-notional via TRADE_SIZE above.
SIZING: SizingConfig | None = None

# Liquidation simulator (cross-margin liq stop + AccountAliveMonitor).
# In this notebook the liq stop acts as a backstop; the protective stop
# below fires first under normal conditions.
LIQUIDATION = LiquidationConfig(
    enabled=True,
    halt_on_account_liquidation=True,
    # mm_rate / fee_rate / min_trade_notional resolved at engine build time.
)

# ─────────────────────────────────────────────────────────────────
# Protective stop loss — the headline knob for this notebook.
# ─────────────────────────────────────────────────────────────────
# STOP_PCT: fraction of entry price at which to place a reduce-only
# protective stop.  Fires before the cross-margin liq stop under normal
# conditions; gives "isolated-margin equivalence" when set to
# 1 / LEVERAGE (so worst-case loss per trade = IM committed).
#
# At LEVERAGE=20 the magic number is 0.05 → $2000 notional × 5% = $100
# loss per trade = $2000 / 20× = IM.  See docs/LIQUIDATION_AND_SIZING.md
# §"Protective stop loss" for the math.
#
# This notebook defaults to ENABLED at 5%; flip to None to compare the
# no-stop baseline.
STOP_PCT: float | None = 0.05

# §4.5 sensitivity sweep grid.  None = baseline (no protective stop, only
# cross-margin liq stop).  Keep this list ordered so the sweep + plot
# read left-to-right from "no stop" through tight → loose.
STOP_PCTS: list[float | None] = [None, 0.02, 0.03, 0.05, 0.075, 0.10, 0.15]

# Save behavior
SAVE_TEARSHEET   = False
SAVE_ON_RUN_ALL  = True


# ─────────────────────────────────────────────────────────────────
# Derived values — computed from the config above.  Don't edit.
# ─────────────────────────────────────────────────────────────────
DATA_CFG       = get_venue_config(DATA_SOURCE)
VENUE_CFG      = get_venue_config(EXEC_VENUE)
TRADE_NOTIONAL = Decimal(TRADE_SIZE)
INSTRUMENT_ID  = make_instrument_id(ASSET, DATA_SOURCE)
TRADING_PAIR   = INSTRUMENT_ID.split("-")[0]
BAR_TYPE_STR   = bar_type_str(INSTRUMENT_ID, BAR_INTERVAL)
VENUE          = Venue(DATA_CFG.nt_venue)


# ─────────────────────────────────────────────────────────────────
# File naming convention
# ─────────────────────────────────────────────────────────────────
# SWEEP_NAME  = strategy + instrument + interval.  Per-notebook prefix
#               "MACross-{MA_TYPE}-SL" distinguishes from the plain
#               ma_cross.ipynb sweep so parquets don't collide.
# RESULT_NAME = SWEEP_NAME + "_f{FAST_MA}_s{SLOW_MA}_sp{STOP_PCT}"
CATALOG_PATH = "../../data/catalog"
SWEEP_NAME   = f"MACross-{MA_TYPE}-SL_{ASSET}_{EXEC_VENUE}_{BAR_INTERVAL}"
_sp_tag      = f"_sp{STOP_PCT}" if STOP_PCT is not None else "_spNone"
RESULT_NAME  = f"{SWEEP_NAME}_f{FAST_MA}_s{SLOW_MA}{_sp_tag}"


print("Imports OK")
print(f"Sizing       : {'equity_frac' if SIZING and SIZING.mode == 'equity_frac' else 'fixed (trade_notional back-compat)'}")
print(f"Liquidation  : {'ENABLED' if LIQUIDATION.enabled else 'disabled'}")
print(f"Stop loss    : {'ENABLED at ' + str(STOP_PCT * 100) + '%' if STOP_PCT else 'disabled (cross-margin liq stop only)'}")
if STOP_PCT is not None:
    _expected_loss = float(TRADE_NOTIONAL) * STOP_PCT
    _im            = float(TRADE_NOTIONAL) / LEVERAGE
    print(f"  Worst-case loss per trade: {_expected_loss:.2f} (IM = {_im:.2f}, ratio = {_expected_loss/_im:.2f}×)")
print(f"MA type      : {MA_TYPE}")
print(f"Grid size    : {len(FAST_PERIODS)}×{len(SLOW_PERIODS)} (fast×slow), "
      f"{sum(1 for f in FAST_PERIODS for s in SLOW_PERIODS if f < s)} valid combos")
print(f"§4.5 sweep   : {len(STOP_PCTS)} stop_pct values "
      f"({sum(1 for sp in STOP_PCTS if sp is not None)} stops + "
      f"{sum(1 for sp in STOP_PCTS if sp is None)} no-stop baseline)")

### 1.2 Load data + resolve liquidation config

Loads instrument and bars from the catalog (with optional date filter), overrides fees with the simulated exec-venue's rates, and resolves the `LIQUIDATION` config's mm_rate / fee_rate / min_trade_notional from `VENUE_CFG` / `SIZING` / instrument metadata.

In [ ]:
instrument, bars = load_backtest_data(
    catalog_path=CATALOG_PATH,
    instrument_id=INSTRUMENT_ID,
    bar_type_str=BAR_TYPE_STR,
    venue_config=VENUE_CFG,           # overrides instrument fees with exec-venue rates
    date_start=DATE_START,
    date_end=DATE_END,
)
CURRENCY = instrument.settlement_currency

# Resolve LIQUIDATION fields (mm_rate, fee_rate, min_trade_notional) from
# VENUE_CFG / SIZING / instrument.  When SIZING is None we synthesize a
# fixed-mode SizingConfig from TRADE_NOTIONAL purely so the resolver has
# a min_trade_notional fallback.
LIQ_RESOLVED = resolve_strategy_liquidation_config(
    user=LIQUIDATION,
    venue_config=VENUE_CFG,
    instrument=instrument,
    sizing=SIZING or SizingConfig(mode="fixed", fixed_notional=TRADE_NOTIONAL),
)

print_setup_summary(
    instrument, bars,
    data_source=DATA_SOURCE, exec_venue=EXEC_VENUE, leverage=LEVERAGE,
)
print_liquidation_resolution(LIQ_RESOLVED, leverage=LEVERAGE)

## 2. Single-config backtest

Configure the BacktestEngine, register the strategy + actors + diagnostics subscribers, run, and pull reports.

### 2.1 Configure engine + venue

Builds a `BacktestEngine` with the instrument + bars + venue, wires up the `AccountAliveMonitor` actor (when liquidation simulation is enabled), and confirms the registered components.

In [ ]:
engine = make_engine(
    VENUE, instrument, bars, STARTING_CAPITAL,
    leverage=LEVERAGE,
    liquidation=LIQ_RESOLVED,
    venue_config=VENUE_CFG,
    sizing=SIZING,
)
print("Engine configured.")

# Confirm AccountAliveMonitor wiring when liquidation is enabled.
if LIQ_RESOLVED is not None and LIQ_RESOLVED.enabled:
    actor_ids = [str(a) for a in engine.kernel.trader.actor_ids()]
    aam = [a for a in actor_ids if "AccountAlive" in a]
    print(f"  Registered actors  : {actor_ids}")
    print(f"  AccountAliveMonitor: {'✓ present' if aam else '✗ MISSING'}")
    print(f"  RiskEngine state   : {engine.kernel.risk_engine.trading_state}")

### 2.2 Subscribe to liquidation events

Registers in-notebook listeners on the `events.liquidation.*` topics so we can verify after the run that the simulator fired as expected.

In [ ]:
position_liquidations: list = []
account_liquidations: list = []

engine.kernel.msgbus.subscribe(
    topic=TOPIC_POSITION_LIQUIDATED,
    handler=position_liquidations.append,
)
engine.kernel.msgbus.subscribe(
    topic=TOPIC_ACCOUNT_LIQUIDATED,
    handler=account_liquidations.append,
)
print("Subscribed to liquidation topics — events will accumulate during run.")

### 2.3 Add strategy + run

Constructs the `MACross` strategy with the chosen parameters, sizing, and (resolved) liquidation config, adds it to the engine, and runs the backtest end-to-end.  Prints liquidation-event counters captured by the in-notebook listener so you can sanity-check that the simulator behaved as expected.

In [ ]:
config = MACrossConfig(
    instrument_id=instrument.id,
    bar_type=BarType.from_str(BAR_TYPE_STR),
    sizing=SIZING,
    trade_notional=TRADE_NOTIONAL,
    ma_type=MA_TYPE,
    fast_period=FAST_MA,
    slow_period=SLOW_MA,
    liquidation=LIQ_RESOLVED,
    stop_pct=STOP_PCT,
)
strategy = MACross(config=config)

# Confirm the strategy received the resolved configs.
_s = strategy._sizing
print(f"Strategy._sizing             : mode={_s.mode}, fixed={_s.fixed_notional}, "
      f"risk_frac={_s.risk_frac}, stop_pct={_s.stop_pct}, "
      f"min_notional={_s.min_notional}")
print(f"Strategy._liq_config         : {strategy._liq_config}")
print(f"Strategy._protective_stop_pct: {strategy._protective_stop_pct}")

engine.add_strategy(strategy)
engine.run()
print()
print("Backtest complete.")

# Post-run state confirmation.
print(f"Post-run RiskEngine state    : {engine.kernel.risk_engine.trading_state}")
print(f"Position liquidations fired  : {len(position_liquidations)}")
print(f"Account liquidations fired   : {len(account_liquidations)}")
print(f"Protective stops fired       : {strategy._protective_count}")
if position_liquidations:
    print()
    print("First 5 position liquidation events:")
    for ev in position_liquidations[:5]:
        print(f"  {ev}")
if account_liquidations:
    print()
    print("Account liquidation event:")
    for ev in account_liquidations:
        print(f"  {ev}")

### 2.4 Reports

Pulls fills, positions, and account-balance reports from the engine for downstream analysis cells.  The orders/positions tables are also useful for quick manual eyeballing.

In [ ]:
fills_report     = engine.trader.generate_order_fills_report()
positions_report = engine.trader.generate_positions_report()
account_report   = engine.trader.generate_account_report(VENUE)

print(f"Order fills : {len(fills_report)}")
print(f"Positions   : {len(positions_report)}")

# Min balance + cross-check with liquidation events.
if not account_report.empty:
    totals = account_report["total"].astype(float)
    min_bal = totals.min()
    print(f"Min balance : {min_bal:.4f}")

    if LIQ_RESOLVED is not None and LIQ_RESOLVED.enabled:
        # With AccountAliveMonitor running, min_balance should not go far
        # below the configured floor. If it does, the actor missed a beat.
        if min_bal < -1.0 and not account_liquidations:
            print(f"⚠️ min_balance={min_bal:.2f} but no AccountLiquidated event "
                  f"fired — actor missed an equity breach.")
        if account_liquidations and min_bal <= 0:
            print(f"  (min_bal ≤ 0 after AccountLiquidated is expected from "
                  f"residual fee debits.)")
    else:
        if min_bal <= 0:
            print(f"\n⚠️ LIQUIDATED — min balance was {min_bal:.2f}")
            print("PnL results after liquidation are meaningless.")

print()
print("=== Recent Fills ===")
display(fills_report.tail(10))

print("\n=== Recent Positions ===")
display(positions_report.tail(10))

### 2.5 Run diagnostics

Standard "what just happened" panel: order-status counts, denied/rejected warnings, balance-curve high/low/final, lowest-N rows for drawdown anatomy, and a margin-field interpretation block.

In [ ]:
print_run_diagnostics(
    engine, VENUE, instrument,
    trade_notional=TRADE_NOTIONAL,
    leverage=LEVERAGE,
)

### 2.6 Calculate analyzer stats

Runs `analyzer.calculate_statistics()` so the analyzer's PnL section (Total PnL, Win Rate, Expectancy, Avg/Max/Min Winner/Loser) is available for the cells below.  The Returns section is deliberately ignored — see the caveat doc.

In [ ]:
account   = engine.cache.account_for_venue(VENUE)
positions = engine.cache.position_snapshots() + engine.cache.positions()
analyzer  = engine.portfolio.analyzer
analyzer.calculate_statistics(account, positions)
print(f"Stats calculated — {len(positions)} positions")

### 2.7 Exit classification (forced-exit + account-liq detection)

Walks every closed position and buckets the closing fill by
`order.tags`: `strategy_exit` / `protective_stop` / `liquidation`.
Also resolves any `AccountLiquidated` event to the open position(s)
that triggered the equity breach.

Both outputs are passed into the price chart, equity-curve chart,
trade-distribution chart, TVLC HTML report, and v2 tearsheet so
forced exits stand out from regular strategy exits.

In [ ]:
exits = classify_position_exits(positions, engine)
acct_liq = find_account_liq_culprit(
    account_liquidations, positions, account_report,
)

_cause = exits['close_cause'] if not exits.empty else None
_n_strat = int((_cause == 'strategy_exit').sum()) if _cause is not None else 0
_n_pstop = int((_cause == 'protective_stop').sum()) if _cause is not None else 0
_n_liq   = int((_cause == 'liquidation').sum()) if _cause is not None else 0
print(
    f"Exits: {_n_strat} strategy / {_n_pstop} protective_stop / "
    f"{_n_liq} position-liquidation"
)
if acct_liq:
    print(
        f"Account liquidated at {acct_liq['liq_ts_iso']} — "
        f"equity {acct_liq['equity_before']} -> {acct_liq['equity_at_liq']} "
        f"(culprit positions: {len(acct_liq['culprit_position_ids'])})"
    )

## 3. Single-config analysis

Charts and tables for the single-config run.  Each cell uses only trustworthy v2 metrics (realized PnL + event-time balances).

### 3.1 NT tearsheet (DISABLED)

NT's `create_tearsheet()` is built on the upstream-broken returns methodology and most of its panels are misleading.  The cell below is a stub explaining what was removed and pointing at the trustworthy equivalents.  See the v2 tearsheet at the end of section 3 for the single-file replacement.

In [ ]:
# Section 3.1 is intentionally a no-op — see the markdown above for
# the rationale.  The trustworthy equivalents to NT's create_tearsheet
# panels live in:
#
#   • PnL stats           → Section 3.5  (print_summary_stats)
#   • Equity + drawdown   → Section 3.3  (plot_equity_curve, event-time)
#   • Drawdown anatomy    → Section 3.4  (plot_drawdown_distribution)
#   • Trade distributions → Section 3.6  (plot_trade_distributions)
#   • Yearly breakdown    → Section 3.7  (plot/print_yearly_breakdown)
#   • Regime breakdown    → Section 3.8  (performance_by_regime)
#   • Single-file archive → Section 3.11 (generate_v2_tearsheet)
#
# When upstream NT lands the daily-MTM fix, uncomment below to restore
# the integrated single-file report.
#
# from nautilus_trader.analysis import create_tearsheet
# html = create_tearsheet(
#     engine,
#     output_path=None,
#     title=f"{MA_TYPE}Cross({FAST_MA}/{SLOW_MA}) | {ASSET} {BAR_INTERVAL}",
# )
# display(HTML(html))
print("Section 3.1 (NT tearsheet) is disabled — see comment above for the")
print("trustworthy equivalents and the upstream caveat.")

### 3.2 Price chart with MA overlay + trade markers

Inline candlestick chart with the strategy's fast/slow MAs and buy/sell markers at fill timestamps.  Useful for eyeballing whether entries/exits look reasonable.

In [ ]:
fig = plot_ma_cross(
    bars, fills_report,
    fast_period=FAST_MA,
    slow_period=SLOW_MA,
    ma_type=MA_TYPE,
    instrument_label=INSTRUMENT_ID,
    bar_label=BAR_INTERVAL,
    exit_classification=exits,
    account_liq_event=acct_liq,
)
fig.show(config=dict(
    modeBarButtonsToRemove=["autoScale2d", "lasso2d", "select2d"],
    displaylogo=False,
))

### 3.3 Equity & drawdown (event-time)

Event-time balance curve with running peak overlay and drawdown band.  **Not** a daily mark-to-market curve — between events the line is flat (last known balance).  Drawdown is real, peak-to-trough on the event-time series.

In [ ]:
plot_equity_curve(
    account_report,
    f"{MA_TYPE}Cross({FAST_MA}/{SLOW_MA})  {INSTRUMENT_ID} {BAR_INTERVAL}",
    currency=CURRENCY,
    exit_classification=exits,
    account_liq_event=acct_liq,
)

### 3.4 Drawdown duration distribution

Aggregates every underwater period in the run into two histograms:
**depth** (% from peak to trough) and **duration** (how long until
recovery, or until the end of the sample).  Long drawdowns kill
psychology much more than deep ones — a 30% drawdown that recovers in
2 weeks is easier to stomach than a 10% drawdown that takes 18 months
to unwind.

In [ ]:
plot_drawdown_distribution(
    account_report,
    title=f"Drawdown distribution — {MA_TYPE}Cross({FAST_MA}/{SLOW_MA})",
    currency=str(CURRENCY),
    bar_interval_ns=int(bars[1].ts_event - bars[0].ts_event) if len(bars) > 1 else None,
)

### 3.5 Summary statistics

Prints the analyzer's General + PnL sections (the trustworthy parts).  Returns section is suppressed.

In [ ]:
print_summary_stats(analyzer, num_positions=len(positions), currency=CURRENCY)

### 3.6 Trade distributions

Three-panel trade-quality view: PnL distribution histogram (wins green, losses red), trade duration histogram (bimodal = two strategies in one), and concentration bars (% of total |PnL| from top/bottom 1, 3, 5 trades — concentration → fragility).

In [ ]:
plot_trade_distributions(
    positions,
    title=f"{MA_TYPE}Cross({FAST_MA}/{SLOW_MA})  {INSTRUMENT_ID} {BAR_INTERVAL}",
    bar_interval_ns=int(bars[1].ts_event - bars[0].ts_event) if len(bars) > 1 else None,
    currency=str(CURRENCY),
    exit_classification=exits,
)

### 3.7 Per-year breakdown

Year-over-year consistency.  Sign-flipping bars or diverging win-rate/profit-factor lines flag a regime-dependent strategy (e.g. only profitable in trending years).

In [ ]:
yearly = performance_by_year(positions, starting_capital=STARTING_CAPITAL)
print_yearly_breakdown(yearly, currency=str(CURRENCY))
plot_yearly_breakdown(
    yearly,
    title=f"Yearly performance — {MA_TYPE}Cross({FAST_MA}/{SLOW_MA})",
    currency=str(CURRENCY),
)

### 3.8 Regime breakdown (ADX-tagged)

Tags every bar as TRENDING vs RANGING via ADX, then splits realized PnL by the regime active at trade open.  Reveals what the strategy is actually being paid for.

In [ ]:
regime_df = tag_regimes(bars, adx_period=14, adx_trending_threshold=25.0)
regime_perf = performance_by_regime(
    positions, regime_df, starting_capital=STARTING_CAPITAL,
)
if not regime_perf.empty:
    display(regime_perf.style.format({
        "pnl": "{:>10,.2f}", "pnl_pct": "{:>6.2f}%",
        "win_rate": "{:.2%}", "profit_factor": "{:>6.2f}",
        "avg_winner": "{:>8,.2f}", "avg_loser": "{:>8,.2f}",
        "avg_duration": "{:>5.1f}h",
    }))
else:
    print("No closed trades or no overlap between trades and regime data.")

### 3.9 Comparison baselines

Two industry-standard "does my strategy actually have alpha?" checks:
1. **Buy & hold (spot)** — would just holding the asset have done better?
2. **Random entry** — does the strategy beat random with the same trade count and average duration?  If not, no entry-timing alpha.

Reports a leveraged-B&H *counterfactual* as a footnote (single perp position at strategy leverage, ignores liquidation — not a realistic benchmark).

In [ ]:
baselines = baselines_for_strategy(
    positions, bars,
    starting_capital=float(STARTING_CAPITAL),
    notional_per_trade=float(TRADE_NOTIONAL),
    fee_rate=float(VENUE_CFG.taker_fee),
    leverage=float(LEVERAGE),
    n_simulations=1000,
    random_seed=42,
)

# Pre-compute the strategy PnL for the verdict + chart + tearsheet.
strategy_pnl = float(analyzer.total_pnl(CURRENCY))

print_baselines_verdict(
    baselines, strategy_pnl=strategy_pnl,
    leverage=LEVERAGE, currency=str(CURRENCY),
)

# Aliases for downstream cells (10f tearsheet, etc.)
bh_spot = baselines["buy_and_hold"]
random_dist = baselines["random_entry"]

plot_baselines_comparison(
    strategy_pnl=strategy_pnl,
    buy_and_hold_pnl=bh_spot["pnl"],
    random_entry_dist=random_dist,
    title=f"Strategy vs spot B&H + random — {MA_TYPE}Cross({FAST_MA}/{SLOW_MA})",
    currency=str(CURRENCY),
)

### 3.10 Bootstrap PnL confidence interval

Resamples the per-trade PnL list with replacement N times, sums each
resample to get a synthetic "total PnL", and reports the distribution
(mean, median, 5/25/75/95 percentiles).  Single-PnL point estimates
are meaningless without dispersion: "PnL = 9,510 with 95% CI
[4,200, 14,800]" tells a much more honest story than just "PnL = 9,510".

**Caveat:** trade-return bootstrap assumes IID trades.  Real strategies
have autocorrelation (winning streaks cluster, drawdowns cluster).  Use
this as a first-pass dispersion estimate — block-bootstrap in
`validate_strategy.ipynb` is more honest.

In [ ]:
closed_pnls = [
    float(p.realized_pnl.as_decimal())
    for p in positions
    if p.is_closed and p.realized_pnl is not None
]
if closed_pnls:
    boot_dist = bootstrap_total_pnl(closed_pnls, n_iterations=10_000, seed=42)
    print(f"Actual total PnL : {boot_dist['actual_total']:>12,.2f}")
    print(f"Bootstrap median : {boot_dist['median']:>12,.2f}")
    print(f"5th–95th pct CI  : "
          f"[{boot_dist['pct_5']:>10,.2f}, {boot_dist['pct_95']:>10,.2f}]")
    print(f"25th–75th pct IQR: "
          f"[{boot_dist['pct_25']:>10,.2f}, {boot_dist['pct_75']:>10,.2f}]")
    plot_bootstrap_pnl(
        boot_dist,
        title=f"Bootstrap PnL — {MA_TYPE}Cross({FAST_MA}/{SLOW_MA})",
        currency=str(CURRENCY),
    )
else:
    print("No closed trades — skipping bootstrap PnL CI.")

### 3.11 v2 tearsheet (single-file shareable archive)

Self-contained HTML report composing everything from sections 3.3–3.8 into one archivable file.  Replaces NT's broken `create_tearsheet`.  Saved to `reports/tearsheets/{RESULT_NAME}_tearsheet_{ts}.html`.

In [ ]:
v2_tearsheet_path = generate_v2_tearsheet(
    positions=positions,
    account_report=account_report,
    bars=bars,
    starting_capital=STARTING_CAPITAL,
    currency=str(CURRENCY),
    instrument_label=INSTRUMENT_ID,
    bar_interval=BAR_INTERVAL,
    strategy_label=f"{MA_TYPE}Cross({FAST_MA}/{SLOW_MA})",
    leverage=LEVERAGE,
    fee_rate=float(VENUE_CFG.taker_fee),
    yearly_df=yearly,
    regime_df=regime_perf,
    baselines={
        "buy_and_hold": bh_spot,
        "random_entry": random_dist,
    },
    strategy_pnl=strategy_pnl,
    filename=f"{RESULT_NAME}_tearsheet",  # snapshot — appends _{ts}.html
    open_browser=True,
    exit_classification=exits,
    account_liq_event=acct_liq,
)
print(f"Saved: {v2_tearsheet_path}")

## 4. Parameter sweep

Grid search over fast/slow MA periods.  Persists results to `data/sweeps/{SWEEP_NAME}.parquet`; a sortable HTML report is generated at the end of the section.

### 4.1 Run sweep

Calls `run_sweep` with the param grid (`FAST_PERIODS` × `SLOW_PERIODS`
filtered to `fast < slow`) plus any `SPOTLIGHT_COMBOS` from cell 1.
Each combo runs in its own isolated `BacktestEngine` with the
liquidation simulator enabled.  Results land in a v2-schema parquet
(see `SWEEP_SCHEMA_VERSION`).

In [ ]:
def ma_factory(eng, params):
    cfg = MACrossConfig(
        instrument_id=instrument.id,
        bar_type=BarType.from_str(BAR_TYPE_STR),
        sizing=SIZING,
        trade_notional=TRADE_NOTIONAL,
        ma_type=MA_TYPE,
        fast_period=params["fast"],
        slow_period=params["slow"],
        liquidation=LIQ_RESOLVED,
        stop_pct=params.get("stop_pct", STOP_PCT),
    )
    eng.add_strategy(MACross(cfg))

grid_combos = [
    {"fast": f, "slow": s}
    for f in FAST_PERIODS for s in SLOW_PERIODS if f < s
]
combos = grid_combos + SPOTLIGHT_COMBOS  # SPOTLIGHT_COMBOS from cell 1

results_df = run_sweep(
    venue=VENUE, instrument=instrument, bars=bars,
    starting_capital=STARTING_CAPITAL,
    param_combos=combos,
    strategy_factory=ma_factory,
    strategy_name=f"MACross-{MA_TYPE}-{EXEC_VENUE}",
    instrument_id=INSTRUMENT_ID,
    bar_interval=BAR_INTERVAL,
    sweep_name=SWEEP_NAME,
    leverage=LEVERAGE,
    liquidation=LIQ_RESOLVED,
    venue_config=VENUE_CFG,
    sizing=SIZING,
)

### 4.2 PnL heatmap

Diverging RdYlGn heatmap coloring `total_pnl` across the fast/slow grid.  Liquidated combos are visually flagged.

In [ ]:
# Save the heatmap PNG so the sweep HTML (cell 4.4) can embed it.
# Path matches the convention used by scripts/batch_backtest.py.
HEATMAP_PATH = f"../../reports/sweeps/{SWEEP_NAME}_heatmap.png"

plot_pnl_heatmap(
    results_df, row_col="slow", col_col="fast",
    row_label=f"Slow {MA_TYPE} Period", col_label=f"Fast {MA_TYPE} Period",
    title=f"Total PnL ({CURRENCY}) by {MA_TYPE} Parameters",
    save_to=HEATMAP_PATH,
)

### 4.3 Liquidation diagnostics

Trustworthiness cross-checks for the simulator's behavior across the full sweep: schema completeness, `min_balance` / `liquidated_account` consistency, halt enforcement (post-halt denials), fee model round-trip cross-check, and slippage distribution (gap risk).

In [ ]:
print_sweep_liquidation_diagnostics(
    results_df,
    liq_resolved=LIQ_RESOLVED,
    trade_notional=TRADE_NOTIONAL,
)

### 4.4 Sortable HTML sweep table

Self-contained interactive HTML report (DataTables.js).  Click headers to sort, search any column, multi-column sort with shift-click, CSV export.  Liquidated rows highlighted red; spotlight rows highlighted gold with a [SPOT] badge.  Saved to `reports/sweeps/{SWEEP_NAME}_sweep.html`.

In [ ]:
sweep_html_path = generate_sweep_html(
    results_df,
    filename=f"{SWEEP_NAME}_sweep",
    open_browser=True,
    heatmap_path=HEATMAP_PATH,
)

display(Markdown(
    f"📊 **Sweep report opened in your default browser.** `{sweep_html_path}`"
))

### 4.5 Stop-loss sensitivity sweep

Picks the best Stage 1 MA combo from §4.1 (excluding liquidated + spotlight rows) and re-runs the strategy at that combo across `STOP_PCTS` — a range of protective stop-loss percentages plus `None` (no protective stop, baseline).

The trade-off the chart reveals:
- **Tight stops** (small stop_pct): cap individual losses but cause more whipsaw exits — fewer winners hit their full move.
- **Loose stops** (large stop_pct): let winners run longer but allow drawdowns to deepen.
- **None baseline**: PnL is whatever the cross-margin liq stop allows (~49.5% adverse before forced exit at default leverage).

Look for the inflection where tightening the stop starts costing more PnL than it saves in drawdown.

In [ ]:
import pandas as _pd
import matplotlib.pyplot as _plt

# Pick best Stage 1 MA combo from §4.1 (exclude liquidated + spotlight).
_s1 = results_df.copy()
if "_kind" in _s1.columns:
    _s1 = _s1[_s1["_kind"] != "spotlight"]
if "liquidated" in _s1.columns:
    _s1 = _s1[~_s1["liquidated"].fillna(False).astype(bool)]
best_row = _s1.loc[_s1["total_pnl"].idxmax()]
best_fast = int(best_row["fast"])
best_slow = int(best_row["slow"])
print(f"Best Stage 1 combo: {MA_TYPE}({best_fast}/{best_slow}) "
      f"PnL={best_row['total_pnl']:,.2f}, max DD={best_row.get('max_drawdown_pct', float('nan')):.2f}%")
print()

# Stage 2: 1D sweep over STOP_PCTS at best MA combo.
stage2_combos = [
    {"fast": best_fast, "slow": best_slow, "stop_pct": sp}
    for sp in STOP_PCTS
]

stop_loss_df = run_sweep(
    venue=VENUE, instrument=instrument, bars=bars,
    starting_capital=STARTING_CAPITAL,
    param_combos=stage2_combos,
    strategy_factory=ma_factory,
    strategy_name=f"MACross-{MA_TYPE}-{EXEC_VENUE}-stop-loss",
    instrument_id=INSTRUMENT_ID,
    bar_interval=BAR_INTERVAL,
    sweep_name=f"{SWEEP_NAME}_stop-loss-sensitivity",
    leverage=LEVERAGE,
    liquidation=LIQ_RESOLVED,
    venue_config=VENUE_CFG,
    sizing=SIZING,
)

# Pretty-print the stop_pct column for the table + chart.
def _fmt_sp(sp: float | None) -> str:
    if sp is None or _pd.isna(sp):
        return "No stop"
    return f"{sp*100:.1f}%"

stop_loss_df = stop_loss_df.copy()
stop_loss_df["stop_pct_label"] = stop_loss_df["stop_pct"].apply(_fmt_sp)

# 2-panel chart: PnL line + max drawdown bars across STOP_PCTS.
fig, (ax1, ax2) = _plt.subplots(2, 1, figsize=(11, 7), sharex=True)
xs = list(range(len(stop_loss_df)))
labels = stop_loss_df["stop_pct_label"].tolist()

ax1.plot(xs, stop_loss_df["total_pnl"], marker="o", color="steelblue", linewidth=2)
ax1.axhline(0, color="grey", linewidth=0.5)
ax1.grid(True, alpha=0.3)
ax1.set_ylabel(f"Total PnL ({CURRENCY})")
ax1.set_title(f"Stop-loss sensitivity at {MA_TYPE}({best_fast}/{best_slow}) — {INSTRUMENT_ID} {BAR_INTERVAL}")

ax2.bar(xs, stop_loss_df["max_drawdown_pct"], color="coral")
ax2.axhline(0, color="grey", linewidth=0.5)
ax2.grid(True, alpha=0.3)
ax2.set_xticks(xs)
ax2.set_xticklabels(labels, rotation=0)
ax2.set_xlabel("STOP_PCT")
ax2.set_ylabel("Max drawdown (%)")
_plt.tight_layout()
_plt.show()

# Comparison table — all rows, key metrics.
display(stop_loss_df[[
    "stop_pct_label", "total_pnl", "total_pnl_pct", "num_positions",
    "win_rate", "max_drawdown_pct", "pnl_profit_factor", "liquidated",
]].rename(columns={"stop_pct_label": "STOP_PCT"}).style.format({
    "total_pnl": "{:>10,.2f}",
    "total_pnl_pct": "{:>6.2f}%",
    "num_positions": "{:>4.0f}",
    "win_rate": "{:.2%}",
    "max_drawdown_pct": "{:>6.2f}%",
    "pnl_profit_factor": "{:>5.2f}",
}))

### 4.6 Close-cause table (single-config)

For the chosen `STOP_PCT` from §1.1 (single-config run from §2), counts how many fills closed via:

- **Strategy exit** — natural cross-back signal (`MARKET` orders submitted by the strategy, no special tag)
- **Protective stop** — reduce-only `STOP_MARKET` tagged `protective_stop` (this notebook's mixin)
- **Liquidation stop** — reduce-only `STOP_MARKET` tagged `liquidation` (the cross-margin backstop from `LiquidationAware`)

Validates that the protective stop is doing the work it should at the chosen `STOP_PCT` and that liquidation acts as a backstop only.  If the liq-stop count is non-zero with `STOP_PCT` set tighter than the liq distance, that's a sign of a stop-fill failure (e.g. gap-through-stop) — investigate.

**Caveat:** entries (BUY/SELL when flat) and strategy exits (`close_all_positions` on cross-back, plus the immediate re-entry on flips) all submit untagged `MARKET` orders.  This cell doesn't try to distinguish entries from exits — it bundles them as "strategy-driven fills" and reports the count.  Use the §2.4 fills_report for finer-grained inspection.

In [ ]:
import pandas as _pd

# Walk all filled orders in the cache; classify by tag.
all_orders = engine.cache.orders()
filled = [o for o in all_orders if o.is_filled]

protective = [o for o in filled if (o.tags or []) and PROTECTIVE_STOP_TAG in o.tags]
liq        = [o for o in filled if (o.tags or []) and "liquidation" in o.tags]
# All other filled orders are entries + strategy-driven exits (untagged MARKETs).
strategy_fills = [o for o in filled if not (o.tags or [])]

n_positions_closed = sum(1 for p in positions if p.is_closed)

cause_df = _pd.DataFrame([
    {
        "Close cause": "Strategy fill (entry or cross-back exit)",
        "Fill count": len(strategy_fills),
        "Notes": "Untagged MARKET orders — entries and strategy exits bundled.",
    },
    {
        "Close cause": "Protective stop fill",
        "Fill count": len(protective),
        "Notes": (
            f"At STOP_PCT = {STOP_PCT} — fired {len(protective)} time(s)."
            if STOP_PCT is not None
            else "STOP_PCT is None — protective stop is disabled."
        ),
    },
    {
        "Close cause": "Liquidation stop fill",
        "Fill count": len(liq),
        "Notes": "Cross-margin liq stop — should be 0 if STOP_PCT < liq distance and no gaps.",
    },
])
display(cause_df)

print()
print(f"Total filled orders : {len(filled)}")
print(f"Closed positions    : {n_positions_closed}")
print(f"Open positions      : {sum(1 for p in positions if not p.is_closed)}")

if STOP_PCT is None:
    print()
    print("ℹ  STOP_PCT is None — protective stops are disabled.  Set STOP_PCT")
    print("   in cell 1.1 (e.g. 0.05 for 20× isolated-margin equivalent) and")
    print("   re-run §2 onwards to see the protective stop in action.")
elif len(protective) == 0:
    print()
    print("⚠  Protective stop is enabled but never fired.  Two possible reasons:")
    print("   1. No losing trades exceeded the stop_pct threshold (the strategy's")
    print("      cross-back exit caught reversals first).")
    print("   2. The dataset / asset isn't volatile enough at this STOP_PCT to")
    print("      generate trips.  Try a tighter STOP_PCT or a more volatile asset.")
elif len(liq) > 0:
    print()
    print(f"⚠  {len(liq)} liquidation-stop fill(s) despite STOP_PCT being set.")
    print("   Most likely a gap-through-stop event (price gapped past the")
    print("   protective stop trigger before it could fire, then continued to")
    print("   the liq distance).  Inspect §2.4 fills_report timestamps to confirm.")

## 5. Interactive trade chart (TVLC)

TradingView Lightweight Charts standalone HTML for the single-config run.  Trade markers + price overlays + drilldown trade list.  Saved to `reports/charts/{RESULT_NAME}_chart_{ts}.html` and auto-opens in the default browser.

In [ ]:
path = generate_backtest_html(
    bars, fills_report, positions_report,
    fast_period=FAST_MA,
    slow_period=SLOW_MA,
    ma_type=MA_TYPE,
    instrument_label=INSTRUMENT_ID,
    bar_label=BAR_INTERVAL,
    starting_capital=STARTING_CAPITAL,
    result_filename=f"{RESULT_NAME}_chart",
    open_browser=True,
    exit_classification=exits,
    account_liq_event=acct_liq,
)

## 6. Robustness checks

Optional sanity checks run *after* the chosen params have proven out on the primary instrument.  Front-runs the questions an experienced trader would ask before committing capital.

### 6.1 Alt-instrument sanity check

Runs the chosen params on 2–4 alternative instruments.  If the strategy is genuinely structural it should at least *not blow up* off-asset.  Disabled by default — uncomment the template inside the cell to enable.

In [ ]:
# Disabled by default — uncomment the block below to enable.
# Tests whether the chosen params generalize off-asset.  If the
# strategy is genuinely structural it should at least *not blow up*
# on alternate instruments; if it does, the params are probably
# overfit to the primary asset's specific structure.
#
# Pick 2-4 alternates with different beta / liquidity tier (e.g. ETH
# for higher-beta majors, SOL for an alt with different correlation
# regime, XRP for a structurally different asset).  All alternates
# are run with the SAME params, leverage, and configs as the primary
# backtest — only the instrument and its bar data change.

# from src.backtesting.analysis import run_alt_instrument_check
#
# alt_instruments = []
# for asset in ["ETH", "SOL", "XRP"]:
#     alt_id = make_instrument_id(asset, DATA_SOURCE)
#     alt_bar_type = bar_type_str(alt_id, BAR_INTERVAL)
#     alt_inst, alt_bars = load_backtest_data(
#         catalog_path=CATALOG_PATH,
#         instrument_id=alt_id,
#         bar_type_str=alt_bar_type,
#         venue_config=VENUE_CFG,
#         date_start=DATE_START,
#         date_end=DATE_END,
#     )
#     alt_instruments.append({
#         "label": asset,
#         "venue": VENUE,
#         "instrument": alt_inst,
#         "bars": alt_bars,
#         "starting_capital": STARTING_CAPITAL,
#     })
#
# alt_df = run_alt_instrument_check(
#     instruments=alt_instruments,
#     params={"fast": FAST_MA, "slow": SLOW_MA, "ma_type": MA_TYPE},
#     strategy_factory=ma_factory,
#     leverage=LEVERAGE,
#     liquidation=LIQ_RESOLVED,
#     venue_config=VENUE_CFG,
#     sizing=SIZING,
# )
# display(alt_df[[
#     "label", "instrument_id", "total_pnl", "total_pnl_pct",
#     "num_positions", "win_rate", "max_drawdown_pct",
#     "pnl_profit_factor", "liquidated",
# ]])

print("Alt-instrument check — see comments in this cell to enable.")
print("Required: catalog data for the alt instruments must already be")
print("fetched (run scripts/fetch_binance_candles.py for the alt assets).")

### 6.2 Fee sensitivity

Re-runs the chosen params at a range of fee levels.  Identifies the breakeven fee threshold and measures margin of safety.  A strategy that dies at 2× current fees has no slack for execution slippage in real markets.

In [ ]:
fee_df = run_fee_sweep(
    venue=VENUE,
    instrument=instrument,
    bars=bars,
    starting_capital=STARTING_CAPITAL,
    params={"fast": FAST_MA, "slow": SLOW_MA, "ma_type": MA_TYPE},
    strategy_factory=ma_factory,
    leverage=LEVERAGE,
    verbose=False,
)
display(fee_df[["fee_bps", "total_pnl", "total_pnl_pct",
                "num_positions", "breakeven"]])
plot_fee_sensitivity(
    fee_df,
    title=f"Fee sensitivity — {MA_TYPE}Cross({FAST_MA}/{SLOW_MA})",
    currency=str(CURRENCY),
)

## 7. Save & cleanup

Optional: snapshot the executed notebook and dispose the engine.

### 7.1 Save snapshot

Single-click "Run All" workflow.  The save cell below copies the
on-disk notebook + an HTML render to `reports/notebooks/backtest/`
and `reports/html/backtest/` using `RESULT_NAME` as the basename.

**For this to work in a single Run All**, your editor must autosave
cells as they finish:

- **VS Code / Cursor:** Settings → search `files.autoSave` → set to
  `afterDelay`.  Default 1000ms is fine.
- **JupyterLab:** autosave is on by default (every 2 minutes — bump
  the frequency in advanced settings if your runs are short).

The snapshot captures every cell's output **except this cell's own
"Saved → ..." message** (the kernel can't autosave a cell while that
cell is still running).  That trade-off is fine — you saw the message
in the editor.

**Without autosave** the snapshot will contain stale outputs from
the previous run.  Either enable autosave, or use the headless
wrapper instead (see below).

**Headless / reproducible alternative:** for CI runs or whenever you
want a fresh-kernel snapshot, drop to a terminal and run:

```bash
./scripts/snapshot-notebook.sh notebooks/backtest/ma_cross_stop_loss.ipynb
```

(or `.ps1` in PowerShell).  This re-executes the notebook headless via
`jupyter nbconvert --execute` and is the only path that captures
cell 7.1's own output too.

In [ ]:
save_notebook_snapshot(
    "ma_cross_stop_loss.ipynb", RESULT_NAME,
    save_on_run_all=SAVE_ON_RUN_ALL,
)

### 7.2 Cleanup

Disposes the engine to release the Rust LogGuard and free the Cython objects.  Re-running cells above this point requires re-creating the engine.

In [ ]:
engine.dispose()
print("Engine disposed.")

## 8. Scratchpad

Empty cell for ad-hoc analysis.  All variables from the cells above
remain in scope (`positions`, `account_report`, `bars`, `engine`,
`analyzer`, `results_df`, `baselines`, etc.).  Feel free to explore
without disturbing the structured sections.

In [ ]:
# Scratchpad — your ad-hoc analysis goes here.